# Agentic AI Systems — Worklet 13
# Research Grant Management System (apniLeap AURA)
## Lab 2 · Tool-Using Agent — Call Data & Compute

**Phase A · Foundations** &nbsp;|&nbsp; **Goal:** Give the agent tools it must call to act, rather than answering from memory

---

**Description**

A grant budget and data assistant that never invents numbers or policies from memory.
It must execute through structured, typed tools:
1. `read_funding_calls`: Reads published agency calls and their guideline constraints.
2. `compute_budget`: Performs precise arithmetic for direct costs, overheads, contingency, and totals.
3. `cap_report`: Audits category percentages (manpower, equipment, travel) against agency limits.
4. `check_eligibility`: Validates PI career stage and institution type against agency requirements.

**Key concepts demonstrated**

- Strict tool contracts (structured inputs and typed dict outputs with result or explicit error).
- Single choke point `_call()` logging tool name, reason, input, and output.
- LangChain `@tool` / `StructuredTool` compatibility wrapper.
- Graceful error handling (honest reporting when records do not exist).

**Assignments included**

- Assignment 1: Notification drafting tool (`draft_budget_alert`) with justified use when caps breach.
- Assignment 2: Three-tool chain: read call -> compute budget -> evaluate caps with non-empty validation.

---

*Runs standalone on Python 3.10+ (Jupyter or Google Colab): no API key, no network, standard library only. Source script: [`scripts/lab02_tool_using_agent.py`](./scripts/lab02_tool_using_agent.py).*

### Cell 1 · Shared institutional seed data

`seed_data.py` is written to disk so the notebook runs standalone. It is the Python mirror of the course project's `backend/src/seed.js`: funding calls, literature corpus, policies, personnel rate card, users and the sanctioned-project portfolio. *(Collapse — shared by all labs.)*

In [1]:
%%writefile seed_data.py
"""
AURA Seed Data & Corpus -- Research Grant Management System
Corresponds to backend/src/seed.js, faculty.js, and projects.js
"""

INSTITUTION = {
    "id": "inst-mist",
    "name": "Meridian Institute of Science and Technology",
    "short": "MIST",
    "type": "university",
    "city": "Pune",
    "country": "IN",
    "researchOffice": "Research & Development Office",
    "signatory": "Dean, Research & Development",
}

USERS = [
    {
        "id": "usr-krishnan",
        "name": "Dr. Uma Mudengudi",
        "email": "uma.mudengudi@mist.edu.in",
        "role": "dean",
        "title": "Dean, Research & Development",
        "department": "Office of Dean R&D",
        "permissions": ["run:create", "run:read:all", "review:approve", "package:view", "audit:read", "version:rollback", "metrics:read"],
    },
    {
        "id": "usr-suneeta-v-budihal",
        "name": "Dr. Suneeta V Budihal",
        "email": "suneeta.v.budihal@mist.edu.in",
        "role": "pi",
        "title": "Associate Professor",
        "department": "SoECE",
        "permissions": ["run:create", "run:read:own", "draft:edit", "package:view"],
    },
    {
        "id": "usr-satyadhyan-chickerur",
        "name": "Dr. Satyadhyan Chickerur",
        "email": "satyadhyan.chickerur@mist.edu.in",
        "role": "pi",
        "title": "Professor & Head",
        "department": "CSE",
        "permissions": ["run:create", "run:read:own", "draft:edit", "package:view"],
    },
    {
        "id": "usr-research-office",
        "name": "Research & Development Office",
        "email": "grants.cell@mist.edu.in",
        "role": "research_office",
        "title": "Grants Cell & Compliance",
        "department": "Office of Dean R&D",
        "permissions": ["run:create", "run:read:all", "review:approve", "package:view", "audit:read", "version:rollback"],
    },
]

RATE_CARD = {
    "institutionId": "inst-mist",
    "overheadPct": 15,
    "contingencyPct": 5,
    "salaryEscalationPct": 5,
    "equipmentMaintenancePct": 4,
    "effectiveFrom": "2026-04-01",
    "note": "Institutional default rates. Agency-published rates for a call always take precedence.",
}

PERSONNEL = [
    {"id": "per-uma", "name": "Dr. Uma Mudengudi", "role": "PI", "title": "Dean (R&D) & Professor", "department": "SoECE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "professor"},
    {"id": "per-suneeta", "name": "Dr. Suneeta V Budihal", "role": "PI", "title": "Faculty Member", "department": "SoECE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "associate"},
    {"id": "per-satyadhyan", "name": "Dr. Satyadhyan Chickerur", "role": "PI", "title": "Faculty Member", "department": "CSE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "associate"},
    {"id": "per-ra", "name": "Research Associate (to be appointed)", "role": "RA", "title": "Research Associate", "department": "Research Office", "monthlyCost": 48000, "defaultEffortPct": 100, "careerStage": "n/a"},
    {"id": "per-phd", "name": "Doctoral Scholar (to be appointed)", "role": "PhD", "title": "Doctoral Scholar", "department": "Research Office", "monthlyCost": 36000, "defaultEffortPct": 100, "careerStage": "n/a"},
]

POLICIES = [
    {
        "id": "pol-overhead",
        "title": "Institutional overhead and costing policy",
        "tags": ["budget", "overhead"],
        "text": "Overheads are charged at the rate published by the funding agency for the call. Where the agency publishes no rate, the institutional default of 15% of total direct costs applies. Overhead may not fund permanent faculty salaries.",
    },
    {
        "id": "pol-ethics",
        "title": "Human subjects research ethics policy",
        "tags": ["compliance", "ethics", "health"],
        "text": "Any proposal involving human participants, identifiable health data or clinical validation requires Institutional Ethics Committee approval before the award is activated. A conditional certificate may be attached at submission and must be converted to full approval before funds are released.",
    },
    {
        "id": "pol-data",
        "title": "Research data protection policy (DPDP-aligned)",
        "tags": ["compliance", "data", "privacy"],
        "text": "Proposals collecting personal data must include a Data Management Plan covering lawful basis, de-identification, storage location, retention period and a named data steward. Cross-border transfer of health data requires a documented risk assessment.",
    },
    {
        "id": "pol-ai",
        "title": "Disclosure of AI-assisted writing",
        "tags": ["compliance", "ai", "authorship"],
        "text": "AI-assisted drafting must be disclosed in the proposal cover declaration and in the institutional submission record. AI-generated text is a draft until a named faculty member reviews and accepts it; AI tools may not be listed as authors or investigators.",
    },
    {
        "id": "pol-coi",
        "title": "Conflict of interest declaration",
        "tags": ["compliance", "coi"],
        "text": "Every investigator must declare financial interests, advisory roles and institutional relationships that could reasonably be perceived to influence the research. Declarations are collected by the Research Office and retained for the audit period.",
    },
    {
        "id": "pol-signoff",
        "title": "Authorised signatory policy",
        "tags": ["compliance", "approval"],
        "text": "Only the Dean, Research & Development, or a delegated Research Office signatory, may authorise submission to a funding agency. The authorising officer must record the decision, the version reviewed, and any conditions attached to approval.",
    },
]

COMPLIANCE_RULES = [
    {"id": "rule-sections", "kind": "sections", "appliesTo": "all", "severity": "blocking", "requirement": "Every section required by the call is present and non-empty.", "evidence": "Call required-section list vs draft section index."},
    {"id": "rule-pages", "kind": "pageLimit", "appliesTo": "all", "severity": "blocking", "requirement": "The narrative stays within the page limit published for the call.", "evidence": "Word count converted at 500 words per page."},
    {"id": "rule-budget-ceiling", "kind": "budgetCeiling", "appliesTo": "all", "severity": "blocking", "requirement": "The requested total does not exceed the agency ceiling for this call.", "evidence": "Budget total vs call award maximum."},
    {"id": "rule-equipment-cap", "kind": "equipmentCap", "appliesTo": "all", "severity": "blocking", "requirement": "Equipment does not exceed the percentage ceiling in the call guidelines.", "evidence": "Equipment subtotal / total requested."},
    {"id": "rule-salary-cap", "kind": "salaryCap", "appliesTo": "all", "severity": "blocking", "requirement": "Manpower costs stay within the call salary ceiling.", "evidence": "Personnel subtotal / total requested."},
    {"id": "rule-overhead", "kind": "overheadRate", "appliesTo": "all", "severity": "blocking", "requirement": "Overhead is charged at exactly the rate published for the call.", "evidence": "Overhead amount vs rate x direct costs."},
    {"id": "rule-ethics", "kind": "document", "docKey": "ethics", "appliesTo": "all", "severity": "blocking", "requirement": "Ethics clearance is attached where the call or the research involves human participants.", "evidence": "Document register vs call requirement."},
    {"id": "rule-coi", "kind": "document", "docKey": "coi", "appliesTo": "all", "severity": "blocking", "requirement": "Conflict-of-interest declarations are attached for all investigators.", "evidence": "Document register vs call requirement."},
    {"id": "rule-dmp", "kind": "document", "docKey": "dmp", "appliesTo": "all", "severity": "blocking", "requirement": "A Data Management Plan is attached where required.", "evidence": "Document register vs call requirement."},
    {"id": "rule-disclosure", "kind": "aiDisclosure", "appliesTo": "all", "severity": "blocking", "requirement": "AI-assisted drafting is disclosed in the submission declaration.", "evidence": "Institutional AI disclosure policy vs package declaration."},
]

DOCUMENT_REGISTER = [
    {"key": "ethics", "label": "Institutional Ethics Committee clearance", "status": "conditional", "note": "Conditional certificate available; full approval pending protocol review."},
    {"key": "coi", "label": "Conflict-of-interest declarations (all investigators)", "status": "attached", "note": "Signed by all investigators."},
    {"key": "dmp", "label": "Data Management Plan", "status": "attached", "note": "Deposited in the institutional DMP registry."},
    {"key": "budget-justification", "label": "Budget justification", "status": "attached", "note": "Generated by the Budget Planning Agent."},
    {"key": "csr-1", "label": "CSR-1 registration and company CSR policy", "status": "pending", "note": "Company secretarial certificate requested."},
    {"key": "consortium-agreement", "label": "Consortium agreement and partner letters", "status": "pending", "note": "Letters of intent collected; agreement under legal review."},
]

CALLS = [
    {
        "id": "call-serb-crg",
        "agency": "SERB",
        "agencyFull": "Science and Engineering Research Board, DST",
        "program": "Core Research Grant (CRG)",
        "title": "SERB Core Research Grant 2026 -- Individual Investigator",
        "fundingType": "government",
        "domains": ["ai-ml", "energy", "materials", "health", "climate"],
        "tags": ["artificial intelligence", "machine learning", "federated learning", "sustainable energy", "materials science", "medical imaging"],
        "deadline": "2026-12-15",
        "budgetMin": 3000000,
        "budgetMax": 7500000,
        "durationMonths": 36,
        "pageLimit": 25,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 45, "equipmentCapPct": 40, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [
            {"id": "call-serb-crg#g1", "section": "submission", "text": "Proposals must be routed through the institutional research office; individual submissions are not accepted."},
            {"id": "call-serb-crg#g2", "section": "budget", "text": "The total request may not exceed INR 75,00,000 for a duration of up to 36 months."},
            {"id": "call-serb-crg#g3", "section": "budget", "text": "Equipment may not exceed 40% of the total requested budget; requests above this ceiling require a written justification from the Head of Institution."},
            {"id": "call-serb-crg#g4", "section": "budget", "text": "Overheads are payable at 15% of total direct costs."},
            {"id": "call-serb-crg#g5", "section": "documents", "text": "A Data Management Plan and a conflict-of-interest declaration are mandatory at submission."},
            {"id": "call-serb-crg#g6", "section": "format", "text": "The narrative may not exceed 25 pages excluding references and the budget annexure."},
        ],
    },
    {
        "id": "call-icmr-adhoc",
        "agency": "ICMR",
        "agencyFull": "Indian Council of Medical Research",
        "program": "Ad-hoc Research Project",
        "title": "ICMR Ad-hoc Research Project 2026 -- Diagnostics & Public Health",
        "fundingType": "government",
        "domains": ["health", "ai-ml", "public-health"],
        "tags": ["diabetic retinopathy", "medical imaging", "rural health", "screening", "federated learning", "digital health", "primary care"],
        "deadline": "2026-12-05",
        "budgetMin": 2000000,
        "budgetMax": 6000000,
        "durationMonths": 36,
        "pageLimit": 25,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 50, "equipmentCapPct": 30, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [
            {"id": "call-icmr-adhoc#g1", "section": "eligibility", "text": "Applicants must hold a regular position in a recognised institution or hospital and demonstrate access to the study population."},
            {"id": "call-icmr-adhoc#g2", "section": "budget", "text": "The total request must fall between INR 20,00,000 and INR 60,00,000; equipment is capped at 30%."},
            {"id": "call-icmr-adhoc#g3", "section": "ethics", "text": "Institutional Ethics Committee approval is mandatory for any study involving human participants or their data."},
            {"id": "call-icmr-adhoc#g4", "section": "data", "text": "A Data Management Plan describing de-identification and retention is required; identifiable health data may not leave the institution without a documented risk assessment."},
            {"id": "call-icmr-adhoc#g5", "section": "deliverables", "text": "The proposal must state measurable clinical or public-health outcomes for the funding period."},
        ],
    },
    {
        "id": "call-tata-csr",
        "agency": "Tata Trusts",
        "agencyFull": "Tata Trusts -- Corporate Social Responsibility",
        "program": "CSR Research Partnership",
        "title": "Tata Trusts CSR Research Partnership 2026 -- Health & Education",
        "fundingType": "industry",
        "domains": ["health", "education", "public-health"],
        "tags": ["community health", "digital health", "education technology", "rural outreach", "screening"],
        "deadline": "2026-11-15",
        "budgetMin": 2500000,
        "budgetMax": 10000000,
        "durationMonths": 24,
        "pageLimit": 20,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital", "ngo"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 8, "salaryCapPct": 55, "equipmentCapPct": 25, "travelCapPct": 10, "contingencyPct": 3},
        "guidelines": [
            {"id": "call-tata-csr#g1", "section": "compliance", "text": "The activity must fall within Schedule VII of the Companies Act 2013 and the implementing partner must hold a valid CSR-1 registration."},
            {"id": "call-tata-csr#g2", "section": "budget", "text": "Grants range from INR 25,00,00,000 to INR 1,00,00,000 over 24 months; overheads are capped at 8%."},
            {"id": "call-tata-csr#g3", "section": "reporting", "text": "Quarterly outcome reporting against agreed indicators is a condition of release of the second tranche."},
        ],
    },
    {
        "id": "call-eu-horizon",
        "agency": "EU Horizon Europe",
        "agencyFull": "European Commission -- Horizon Europe",
        "program": "Cluster 4: Digital, Industry and Space",
        "title": "Horizon Europe 2027 -- Trustworthy AI for Health Systems (RIA)",
        "fundingType": "international",
        "domains": ["ai-ml", "health", "digital"],
        "tags": ["trustworthy ai", "federated learning", "health systems", "digital health", "data governance", "open science"],
        "deadline": "2027-01-20",
        "budgetMin": 18000000,
        "budgetMax": 45000000,
        "durationMonths": 36,
        "pageLimit": 45,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital"], "consortiumRequired": True, "minPartners": 3, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 25, "salaryCapPct": 40, "equipmentCapPct": 30, "travelCapPct": 15, "contingencyPct": 0},
        "guidelines": [
            {"id": "call-eu-horizon#g1", "section": "eligibility", "text": "Proposals require a consortium of at least three independent legal entities from three different countries."},
            {"id": "call-eu-horizon#g2", "section": "budget", "text": "Personnel costs may not exceed 40% and equipment 30% of the total eligible cost; a 25% flat rate covers indirect costs."},
            {"id": "call-eu-horizon#g3", "section": "open-science", "text": "Open science practices, FAIR data management and a gender and inclusion plan are evaluated under the impact criterion."},
        ],
    },
    {
        "id": "call-serb-supra-expired",
        "agency": "SERB",
        "agencyFull": "Science and Engineering Research Board, DST",
        "program": "SUPRA",
        "title": "SERB SUPRA 2026 -- Closed",
        "fundingType": "government",
        "domains": ["ai-ml", "health"],
        "tags": ["artificial intelligence", "medical imaging"],
        "deadline": "2026-06-30",
        "budgetMin": 1000000,
        "budgetMax": 3000000,
        "durationMonths": 24,
        "status": "closed",
        "verified": True,
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university"], "consortiumRequired": False, "minPartners": 0},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 45, "equipmentCapPct": 40, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [{"id": "call-serb-supra-expired#g1", "section": "status", "text": "This call closed on 30 June 2026 and is retained for reference only."}],
    },
    {
        "id": "call-unverified-sample",
        "agency": "Unknown Aggregator",
        "agencyFull": "Third-party opportunity aggregator",
        "program": "Sample Aggregator Call",
        "title": "Unverified Grant Opportunity 2027",
        "fundingType": "government",
        "domains": ["engineering"],
        "tags": ["engineering"],
        "deadline": "2027-02-01",
        "budgetMin": 800000,
        "budgetMax": 2500000,
        "durationMonths": 24,
        "verified": False,
        "status": "unverified",
        "eligibility": {"careerStages": ["assistant"], "institutionTypes": ["college"], "consortiumRequired": False, "minPartners": 0},
        "budgetRules": {"overheadPct": 10, "salaryCapPct": 40, "equipmentCapPct": 35, "travelCapPct": 12, "contingencyPct": 5},
        "guidelines": [{"id": "call-unverified#g1", "section": "status", "text": "Listing could not be confirmed against an official agency notification."}],
    },
]

LITERATURE = [
    {
        "id": "lit-001",
        "title": "Federated learning for diabetic retinopathy screening across rural clinics",
        "authors": ["S. Banerjee", "R. Iyer", "M. Kulkarni"],
        "year": 2024,
        "venue": "Journal of Medical AI Research",
        "tags": ["federated learning", "diabetic retinopathy", "rural health", "medical imaging", "screening"],
        "abstract": "A four-site federated study reports comparable sensitivity to centralised training while keeping 1.2 million fundus images inside participating clinics.",
        "citations": 61,
        "doi": "10.5555/aura.demo.001",
    },
    {
        "id": "lit-002",
        "title": "Deep learning detection of referable diabetic retinopathy in primary care",
        "authors": ["L. Fernandes", "A. Gupta"],
        "year": 2023,
        "venue": "Ophthalmology Informatics",
        "tags": ["diabetic retinopathy", "medical imaging", "primary care", "screening", "deep learning"],
        "abstract": "A prospective primary-care evaluation of 8,400 patients reports 92% sensitivity and 88% specificity for referable retinopathy.",
        "citations": 148,
        "doi": "10.5555/aura.demo.002",
    },
    {
        "id": "lit-003",
        "title": "Privacy-preserving federated learning in healthcare: a systematic review",
        "authors": ["H. Park", "J. Okafor", "N. Singh"],
        "year": 2024,
        "venue": "Health Data Science Review",
        "tags": ["federated learning", "privacy", "health systems", "data governance"],
        "abstract": "Reviews 74 studies and finds that differential privacy and secure aggregation are rarely combined in clinical deployments.",
        "citations": 96,
        "doi": "10.5555/aura.demo.003",
    },
    {
        "id": "lit-004",
        "title": "Non-mydriatic fundus photography in resource-limited settings",
        "authors": ["P. Adjei", "T. Nguyen"],
        "year": 2022,
        "venue": "Global Eye Health",
        "tags": ["rural health", "screening", "medical imaging", "primary care"],
        "abstract": "Image quality is the dominant failure mode for non-mydriatic cameras operated by trained community health workers.",
        "citations": 57,
        "doi": "10.5555/aura.demo.004",
    },
    {
        "id": "lit-005",
        "title": "Teleophthalmology screening pathways in low-resource districts",
        "authors": ["K. Rao", "D. Bhattacharya"],
        "year": 2023,
        "venue": "Indian Journal of Public Health",
        "tags": ["screening", "rural health", "health systems", "telemedicine"],
        "abstract": "Hub-and-spoke teleophthalmology reduced time-to-diagnosis from 26 to 9 days across six districts.",
        "citations": 73,
        "doi": "10.5555/aura.demo.005",
    },
]

HISTORY = [
    {
        "id": "hist-retina-2024",
        "title": "AI-assisted retinal screening for diabetic retinopathy in Pune district",
        "agency": "ICMR",
        "year": 2024,
        "outcome": "not-funded",
        "amount": 5600000,
        "tags": ["diabetic retinopathy", "screening", "rural health", "medical imaging"],
        "abstract": "Proposed a three-site pilot of automated retinal grading with referral support for 12,000 patients.",
        "lessons": "Reviewers questioned the missing ethics clearance and the absence of a site-level data-governance plan. Both are now prerequisites.",
    },
    {
        "id": "hist-fl-testbed-2023",
        "title": "Federated learning testbed for multi-hospital medical imaging",
        "agency": "SERB",
        "year": 2023,
        "outcome": "funded",
        "amount": 6200000,
        "tags": ["federated learning", "medical imaging", "privacy", "testbed"],
        "abstract": "Built a three-hospital federated learning testbed with secure aggregation and site-level audit logging.",
        "lessons": "Delivered the testbed; clinical validation was out of scope and remains the open gap.",
    },
]


Writing seed_data.py


### Cell 2 · Imports and fixtures

In [2]:
import json
from typing import Dict, Any, List, Optional
from seed_data import CALLS, PERSONNEL, RATE_CARD

### Cell 3 · `read_funding_call` (function)

In [3]:
# ==============================================================================
# Core Typed Tool Implementations
# ==============================================================================

def read_funding_call(call_id: str) -> Dict[str, Any]:
    """Tool: Reads an agency call and its guidelines from the database."""
    for call in CALLS:
        if call["id"] == call_id:
            return {"result": call}
    return {"error": f"Funding call '{call_id}' not found in verified registry"}

### Cell 4 · `read_personnel_rate` (function)

In [4]:
def read_personnel_rate(role: str) -> Dict[str, Any]:
    """Tool: Reads standard salary/rate card for research staff."""
    for person in PERSONNEL:
        if person["role"].lower() == role.lower():
            return {
                "result": {
                    "role": person["role"],
                    "monthlyCost": person.get("monthlyCost", 0),
                    "title": person.get("title", ""),
                }
            }
    return {"error": f"No standard rate found for staff role '{role}'"}

### Cell 5 · `compute_budget` (function)

In [5]:
def compute_budget(lines: List[Dict[str, Any]], rules: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """
    Tool: Performs exact financial arithmetic. Never done in LLM prose.
    Computes direct costs, overhead, contingency, and final total.
    """
    if not lines:
        return {"error": "No budget lines provided"}

    rules = rules or {}
    overhead_pct = float(rules.get("overheadPct", RATE_CARD["overheadPct"]))
    contingency_pct = float(rules.get("contingencyPct", RATE_CARD["contingencyPct"]))

    category_sums = {"personnel": 0, "equipment": 0, "consumables": 0, "travel": 0}
    for line in lines:
        cat = line.get("category", "consumables").lower()
        amt = float(line.get("amount", 0))
        if cat in category_sums:
            category_sums[cat] += amt
        else:
            category_sums["consumables"] += amt

    direct = sum(category_sums.values())
    overhead = round((direct * overhead_pct) / 100.0)
    contingency = round((direct * contingency_pct) / 100.0)
    total = direct + overhead + contingency

    return {
        "result": {
            "categories": category_sums,
            "direct": direct,
            "overhead": overhead,
            "overheadPct": overhead_pct,
            "contingency": contingency,
            "contingencyPct": contingency_pct,
            "total": total,
            "lineCount": len(lines),
        }
    }

### Cell 6 · `cap_report` (function)

In [6]:
def cap_report(lines: List[Dict[str, Any]], rules: Dict[str, Any]) -> Dict[str, Any]:
    """
    Tool: Verifies category amounts against agency percentage ceilings.
    """
    b_res = compute_budget(lines, rules)
    if "error" in b_res:
        return b_res

    budget = b_res["result"]
    total = budget["total"]
    if total <= 0:
        return {"error": "Budget total is zero or negative"}

    caps = [
        {"key": "salary", "label": "Manpower", "category": "personnel", "capPct": float(rules.get("salaryCapPct", 100))},
        {"key": "equipment", "label": "Equipment", "category": "equipment", "capPct": float(rules.get("equipmentCapPct", 100))},
        {"key": "travel", "label": "Travel", "category": "travel", "capPct": float(rules.get("travelCapPct", 100))},
    ]

    report = []
    any_breach = False
    for c in caps:
        amt = budget["categories"].get(c["category"], 0)
        actual_pct = round((amt / total) * 100.0, 1)
        limit_amt = round((c["capPct"] / 100.0) * total)
        status = "over" if actual_pct > (c["capPct"] + 0.05) else "ok"
        if status == "over":
            any_breach = True
        report.append({
            "key": c["key"],
            "label": c["label"],
            "amount": amt,
            "actualPct": actual_pct,
            "capPct": c["capPct"],
            "limitAmount": limit_amt,
            "status": status,
        })

    return {"result": {"caps": report, "anyBreach": any_breach}}

### Cell 7 · `check_eligibility` (function)

In [7]:
def check_eligibility(call_id: str, profile: Dict[str, Any]) -> Dict[str, Any]:
    """
    Tool: Checks career stage and institutional eligibility.
    """
    call_res = read_funding_call(call_id)
    if "error" in call_res:
        return call_res

    call = call_res["result"]
    elig = call.get("eligibility", {})
    career_stages = elig.get("careerStages", [])
    inst_types = elig.get("institutionTypes", [])

    pi_stage = profile.get("careerStage", "associate")
    inst_type = profile.get("institutionType", "university")

    stage_ok = pi_stage in career_stages
    inst_ok = inst_type in inst_types

    return {
        "result": {
            "eligible": stage_ok and inst_ok,
            "checks": [
                {"field": "careerStage", "candidate": pi_stage, "allowed": career_stages, "pass": stage_ok},
                {"field": "institutionType", "candidate": inst_type, "allowed": inst_types, "pass": inst_ok},
            ]
        }
    }

### Cell 8 · `LangChainToolAdapter` (class)

In [8]:
# ==============================================================================
# LangChain Tool Compatibility Adapter
# (Demonstrates how these tools seamlessly wrap into LangChain format)
# ==============================================================================

class LangChainToolAdapter:
    """
    Adapter demonstrating standard LangChain BaseTool / StructuredTool pattern
    without requiring third-party library installation.
    """
    def __init__(self, name: str, description: str, func, args_schema: Optional[Dict] = None):
        self.name = name
        self.description = description
        self.func = func
        self.args_schema = args_schema or {}

    def run(self, **kwargs):
        return self.func(**kwargs)

    def to_langchain_definition(self) -> Dict[str, Any]:
        return {
            "name": self.name,
            "description": self.description,
            "parameters": self.args_schema,
        }


# ==============================================================================
# Agent Implementation with Single Choke Point
# ==============================================================================

TOOLS = {
    "read_funding_call": read_funding_call,
    "read_personnel_rate": read_personnel_rate,
    "compute_budget": compute_budget,
    "cap_report": cap_report,
    "check_eligibility": check_eligibility,
}

### Cell 9 · `GrantDataAssistant` (class)

In [9]:
class GrantDataAssistant:
    def __init__(self):
        self.trace: List[Dict[str, Any]] = []

    def _call(self, tool_name: str, reason: str, **kwargs) -> Any:
        """
        The single choke point through which every tool invocation passes.
        Logs inputs, outputs, timestamps, and asserts output validity.
        """
        if tool_name not in TOOLS:
            raise ValueError(f"Unknown tool '{tool_name}'")

        result = TOOLS[tool_name](**kwargs)
        entry = {
            "tool": tool_name,
            "reason": reason,
            "input": kwargs,
            "output": result,
        }
        self.trace.append(entry)

        if "error" in result:
            raise RuntimeError(f"Tool '{tool_name}' failed: {result['error']}")
        return result["result"]

    def answer_question(self, question: str) -> str:
        """Processes questions requiring traceable tool execution."""
        q = question.lower()

        # Question 1: Budget computation
        if "budget" in q and "total" in q:
            # Sample proposed budget items from faculty
            sample_lines = [
                {"category": "personnel", "item": "Research Associate (3 yrs)", "amount": 1728000},
                {"category": "equipment", "item": "GPU Workstation", "amount": 1800000},
                {"category": "consumables", "item": "Data Storage & APIs", "amount": 600000},
                {"category": "travel", "item": "Clinic Visits", "amount": 400000},
            ]
            computed = self._call(
                "compute_budget",
                "Faculty submitted 4 budget lines; calculating total direct cost and statutory overhead",
                lines=sample_lines,
                rules={"overheadPct": 15, "contingencyPct": 5},
            )
            return (
                f"Computed Budget: Direct = INR {computed['direct']:,}, "
                f"Overhead (15%) = INR {computed['overhead']:,}, "
                f"Contingency (5%) = INR {computed['contingency']:,}, "
                f"Grand Total = INR {computed['total']:,}."
            )

        # Question 2: Funding Call Lookup
        if "guideline" in q or "serb" in q:
            call_data = self._call(
                "read_funding_call",
                "Looking up published guidelines and ceilings for SERB CRG call",
                call_id="call-serb-crg",
            )
            return (
                f"Call '{call_data['title']}' has ceiling INR {call_data['budgetMax']:,} "
                f"and deadline {call_data['deadline']}."
            )

        # Question 3: Graceful failure path
        if "non-existent" in q or "missing" in q:
            try:
                self._call(
                    "read_funding_call",
                    "Attempting lookup of unverified agency identifier",
                    call_id="call-fake-999",
                )
            except RuntimeError as e:
                return f"Graceful failure reported: {e}"

        return "I do not have a configured tool path for that question."

### Cell 10 · `notification_tool` (function)

In [10]:
# ==============================================================================
# Graded Assignments
# ==============================================================================

def notification_tool(recipient: str, alert_message: str) -> Dict[str, Any]:
    """Assignment 1 Tool: Drafts a notification without external side effects."""
    return {
        "result": {
            "draft_id": "ALERT-001",
            "to": recipient,
            "message": alert_message,
            "status": "DRAFTED_NOT_SENT",
        }
    }

TOOLS["send_notification"] = notification_tool

### Cell 11 · `run_assignment_1_notification` (function)

In [11]:
def run_assignment_1_notification(assistant: GrantDataAssistant):
    """
    Assignment 1 -- Add Notification Tool with Justified Use:
    When equipment exceeds the agency cap, call send_notification to draft an alert.
    """
    print("\n--- [Assignment 1] Add Notification Tool with Justified Use ---")
    breaching_lines = [
        {"category": "personnel", "item": "RA", "amount": 1000000},
        {"category": "equipment", "item": "Supercomputer Cluster", "amount": 4500000},  # Breaches 40% cap
        {"category": "consumables", "item": "Reagents", "amount": 500000},
    ]
    caps = assistant._call(
        "cap_report",
        "Evaluating category percentages against SERB CRG limits",
        lines=breaching_lines,
        rules={"salaryCapPct": 45, "equipmentCapPct": 40, "travelCapPct": 10},
    )

    if caps["anyBreach"]:
        equip_cap = next(c for c in caps["caps"] if c["key"] == "equipment")
        alert = assistant._call(
            "send_notification",
            f"Equipment allocation ({equip_cap['actualPct']}%) exceeds {equip_cap['capPct']}% ceiling; drafting warning for PI",
            recipient="suneeta.v.budihal@mist.edu.in",
            alert_message=f"Equipment cost of INR {equip_cap['amount']:,} exceeds agency ceiling.",
        )
        print(f"Notification drafted successfully: {alert}")

### Cell 12 · `run_assignment_2_three_tool_chain` (function)

In [12]:
def run_assignment_2_three_tool_chain(assistant: GrantDataAssistant):
    """
    Assignment 2 -- A Three-Tool Chain:
    1. Read funding call
    2. Check PI eligibility
    3. Calculate budget against call ceiling and caps
    """
    print("\n--- [Assignment 2] Three-Tool Chain Execution ---")
    # Step 1: Read call
    call = assistant._call(
        "read_funding_call",
        "Step 1: Fetching ICMR Ad-hoc call parameters",
        call_id="call-icmr-adhoc",
    )
    # Step 2: Check eligibility
    elig = assistant._call(
        "check_eligibility",
        "Step 2: Checking associate professor eligibility against ICMR requirements",
        call_id="call-icmr-adhoc",
        profile={"careerStage": "associate", "institutionType": "university"},
    )
    # Step 3: Compute budget using call-specific overhead rules
    budget_lines = [
        {"category": "personnel", "amount": 2000000},
        {"category": "equipment", "amount": 1500000},
        {"category": "consumables", "amount": 500000},
    ]
    budget = assistant._call(
        "compute_budget",
        "Step 3: Calculating total cost using call's 15% overhead rule",
        lines=budget_lines,
        rules=call["budgetRules"],
    )
    print(f"3-Tool Chain Result: Call = {call['id']}, Eligible = {elig['eligible']}, Total = INR {budget['total']:,}")

### Cell 13 · Run the lab

The lab demo and both graded assignments, with the console captured for the checks below.

In [13]:
# Console capture: everything the lab prints is shown below AND kept in memory,
# so the acceptance checks in the last cell can assert on the real run.
import io
import sys


class ConsoleTee(io.StringIO):
    """Mirror stdout into a buffer while the lab runs."""

    def __enter__(self):
        self._stdout = sys.stdout
        sys.stdout = self
        return self

    def __exit__(self, *exc):
        sys.stdout = self._stdout
        return False

    def write(self, text):
        self._stdout.write(text)
        return super().write(text)


console = ConsoleTee()

with console:
    print("======================================================================")
    print("Lab 2: Tool-Using Agent -- Call Data & Compute")
    print("======================================================================")

    assistant = GrantDataAssistant()

    q1 = "What is the total budget for the proposed items?"
    print(f"\nQ: {q1}")
    print(f"A: {assistant.answer_question(q1)}")

    q2 = "What are the guidelines for SERB CRG?"
    print(f"\nQ: {q2}")
    print(f"A: {assistant.answer_question(q2)}")

    q3 = "Check status of non-existent call call-fake-999"
    print(f"\nQ: {q3}")
    print(f"A: {assistant.answer_question(q3)}")

    # Run assignments
    run_assignment_1_notification(assistant)
    run_assignment_2_three_tool_chain(assistant)

    # Full execution trace
    print("\n=== Full Tool Call Trace ===")
    for i, t in enumerate(assistant.trace, 1):
        print(f"{i}. tool={t['tool']} | reason=\"{t['reason']}\"")
        print(f"   input: {t['input']}")
        print(f"   output: {t['output']}")

Lab 2: Tool-Using Agent -- Call Data & Compute

Q: What is the total budget for the proposed items?
A: Computed Budget: Direct = INR 4,528,000.0, Overhead (15%) = INR 679,200, Contingency (5%) = INR 226,400, Grand Total = INR 5,433,600.0.

Q: What are the guidelines for SERB CRG?
A: Call 'SERB Core Research Grant 2026 -- Individual Investigator' has ceiling INR 7,500,000 and deadline 2026-12-15.

Q: Check status of non-existent call call-fake-999
A: Graceful failure reported: Tool 'read_funding_call' failed: Funding call 'call-fake-999' not found in verified registry

--- [Assignment 1] Add Notification Tool with Justified Use ---
Notification drafted successfully: {'draft_id': 'ALERT-001', 'to': 'suneeta.v.budihal@mist.edu.in', 'message': 'Equipment cost of INR 4,500,000.0 exceeds agency ceiling.', 'status': 'DRAFTED_NOT_SENT'}

--- [Assignment 2] Three-Tool Chain Execution ---
3-Tool Chain Result: Call = call-icmr-adhoc, Eligible = True, Total = INR 4,800,000.0

=== Full Tool Call Tr

### Cell 14 · Acceptance checks and deliverables

Each check asserts on the trace of the run above. The trace and the check report are exported to `lab2_artifacts/` (committed under `Course-Project/evidence/`).

In [14]:
import json
import re
from pathlib import Path

LAB = 2
CHECKS = [('Budget total is computed by a tool, not asserted', 'Grand Total = INR 5,433,600', 1),
    ('Unknown call id fails gracefully through the tool layer', 'Graceful failure reported', 1),
    ('Notification tool drafts but never sends', 'DRAFTED_NOT_SENT', 1),
    ('Three-tool chain completes with an eligibility verdict', '3-Tool Chain Result: .*Eligible = True', 1),
    ('Every tool call is traced with a reason', '^\\d+\\. tool=\\w+ \\| reason=', 8),
    ('Eligibility is checked by its own typed tool', 'tool=check_eligibility', 1)]

trace = console.getvalue()
universal = [
    ("Run produced console output", r"\S", 1),
    ("Graded Assignment 1 executed", r"\[Assignment 1\]", 1),
    ("Graded Assignment 2 executed", r"\[Assignment 2\]", 1),
]
results = []
for label, pattern, minimum in universal + CHECKS:
    found = len(re.findall(pattern, trace, flags=re.MULTILINE))
    results.append({"check": label, "pattern": pattern, "expected_min": minimum,
                    "found": found, "passed": found >= minimum})
no_errors = "Traceback (most recent call last)" not in trace
results.append({"check": "No unhandled exception in the run", "pattern": "Traceback",
                "expected_min": 0, "found": 0 if no_errors else 1, "passed": no_errors})

for i, r in enumerate(results, 1):
    print(f"[{'PASS' if r['passed'] else 'FAIL'}] {i:>2}. {r['check']}")
passed = sum(r["passed"] for r in results)
print(f"\nLab {LAB}: {passed}/{len(results)} acceptance checks passed")

outdir = Path(f"lab{LAB}_artifacts")
outdir.mkdir(exist_ok=True)
(outdir / f"lab{LAB}_console_trace.txt").write_text(trace, encoding="utf-8")
(outdir / f"lab{LAB}_acceptance_checks.json").write_text(
    json.dumps({"lab": LAB, "passed": passed, "total": len(results), "checks": results}, indent=2),
    encoding="utf-8")
print("Artifacts written to", outdir.as_posix())

assert passed == len(results), "acceptance checks failed"

[PASS]  1. Run produced console output
[PASS]  2. Graded Assignment 1 executed
[PASS]  3. Graded Assignment 2 executed
[PASS]  4. Budget total is computed by a tool, not asserted
[PASS]  5. Unknown call id fails gracefully through the tool layer
[PASS]  6. Notification tool drafts but never sends
[PASS]  7. Three-tool chain completes with an eligibility verdict
[PASS]  8. Every tool call is traced with a reason
[PASS]  9. Eligibility is checked by its own typed tool
[PASS] 10. No unhandled exception in the run

Lab 2: 10/10 acceptance checks passed
Artifacts written to lab2_artifacts
